# SI Figure S1: accuracy-vs-cost Pareto frontier (proton and carbon)

Figure 2A for both nuclei in CDCl3: fitting RMSE vs total compute time, colored by method type, shaped by geometry source, sized by basis (panel A proton, B carbon).

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import delta22
import pareto_plot
import paths

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
dft = delta22.load_query_df_dft(DELTA22_HDF5, XLSX, verbose=False)
nn = delta22.load_query_df_nn(DELTA22_HDF5, XLSX, verbose=False)
dft_gas_timings, nn_timings = delta22.load_pareto_timings(DELTA22_HDF5)

points = delta22.fig2a_pareto_points(dft, nn, dft_gas_timings, nn_timings, n_splits=100)
print(points["nmr_method"].nunique(), "methods;",
      "MagNET total time", float(points.query("nmr_method=='MagNET'")["total_time"].iloc[0]), "s")

## Proton and carbon Pareto panels

Panel A is the proton frontier, B the carbon; they differ only in y-range, the NMR reference point
(wp04 for ¹H, wb97xd for ¹³C), and label offsets.

In [ ]:
method_legend_label_map = {
    "MagNET-Zero": "MagNET-Zero", "ab initio": r"$\it{ab\ initio}$", "double hybrids": "double hybrid",
    "NMR-specific": "NMR-specific", "DFT": "DFT",
}
geometry_legend_label_map = {"pbe0_tz": "DFT (PBE0/cc-pVTZ)", "aimnet2": "ML (AIMNet2)"}
method_color_map = {"ab initio": "#53585f", "MagNET-Zero": "#000000", "double hybrids": "#b3a560",
                    "NMR-specific": "#a72608", "DFT": "#4a8075"}
geometry_marker_map = {"aimnet2": "x", "pbe0_tz": "o"}

# which points to draw in each panel: MagNET-Zero, all PBE0-geometry DFT methods, plus one
# AIMNet2-geometry NMR-specific reference (wp04 for proton, wb97xd for carbon)
query_str = ("nucleus=='{nucleus}' and solvent == 'chloroform' and ("
             " (geometry_type=='aimnet2' and nmr_method.str.startswith('MagNET')) or"
             " (geometry_type=='pbe0_tz' and not nmr_method.str.startswith('MagNET')) or"
             " (geometry_type=='aimnet2' and nmr_method=='wp04' and basis=='pcSseg2' and nucleus=='H') or"
             " (geometry_type=='aimnet2' and nmr_method=='wb97xd' and basis=='pcSseg2' and nucleus=='C'))")

proton_label_positions = {
    ("MagNET-Zero", "N/A", "aimnet2"): {"xytext": (-28, 14)},
    ("wp04", "pcSseg2", "aimnet2"): {"xytext": (6, 4)},
    ("pbe0", "pcSseg1", "pbe0_tz"): {"xytext": (-22, -2)},
    ("b2gp_plyp", "pcSseg1", "pbe0_tz"): {"xytext": (4, 6)},
    ("revdsd_pbep86", "pcSseg1", "pbe0_tz"): {"xytext": (6, -4)},
    ("revdsd_pbep86", "pcSseg2", "pbe0_tz"): {"xytext": (6, 6)},
    ("tpsstpss", "pcSseg2", "pbe0_tz"): {"xytext": (6, -2)},
    ("b97d3", "pcSseg3", "pbe0_tz"): {"xytext": (-28, -2)},
    ("mpw2plyp", "pcSseg3", "pbe0_tz"): {"xytext": (8, 2)},
    ("b2plyp", "pcSseg3", "pbe0_tz"): {"xytext": (8, 0)},
    ("tpsstpss", "pcSseg3", "pbe0_tz"): {"xytext": (6, -12)},
}
carbon_label_positions = {
    ("MagNET-Zero", "N/A", "aimnet2"): {"xytext": (-28, 14)},
    ("wb97xd", "pcSseg2", "aimnet2"): {"xytext": (6, -4)},
    ("b2gp_plyp", "pcSseg1", "pbe0_tz"): {"xytext": (-20, 8)},
    ("revdsd_pbep86", "pcSseg1", "pbe0_tz"): {"xytext": (0, -12)},
    ("hf", "pcSseg1", "pbe0_tz"): {"xytext": (6, -6)},
    ("revdsd_pbep86", "pcSseg2", "pbe0_tz"): {"xytext": (6, 0)},
    ("tpsstpss", "pcSseg2", "pbe0_tz"): {"xytext": (6, -4)},
    ("mpw2plyp", "pcSseg3", "pbe0_tz"): {"xytext": (8, 2)},
    ("dsd_pbep86", "pcSseg3", "pbe0_tz"): {"xytext": (8, 2)},
    ("revdsd_pbep86", "pcSseg3", "pbe0_tz"): {"xytext": (8, -6)},
    ("m062x", "pcSseg3", "pbe0_tz"): {"xytext": (8, 0)},
    ("wp04", "pcSseg3", "pbe0_tz"): {"xytext": (8, 0)},
}

In [ ]:
pareto_plot.plot_pareto_panel(
    points, query_str, nucleus="H", suptitle="MagNET Extends a Flat Pareto Frontier",
    method_color_map=method_color_map, method_legend_label_map=method_legend_label_map,
    geometry_legend_label_map=geometry_legend_label_map, geometry_marker_map=geometry_marker_map,
    xlim_left=(1.7, 1.8), xlim_right=(3.8, 5.5), ylim=(0.08, 0.3),
    figsize=(12, 6), marker_alpha=0.8, manual_label_positions=proton_label_positions,
    save_png=figure_path("si_figure_s01_pareto_1H.png"),
)

pareto_plot.plot_pareto_panel(
    points, query_str, nucleus="C", suptitle="MagNET Extends a Flat Pareto Frontier",
    method_color_map=method_color_map, method_legend_label_map=method_legend_label_map,
    geometry_legend_label_map=geometry_legend_label_map, geometry_marker_map=geometry_marker_map,
    xlim_left=(1.7, 1.8), xlim_right=(3.8, 5.5), ylim=(1.2, 4.2),
    figsize=(12, 6), marker_alpha=0.8, manual_label_positions=carbon_label_positions,
    save_png=figure_path("si_figure_s01_pareto_13C.png"),
)